# Robodog Natural-Language LLM Control (Google Colab)

Welcome! In this notebook you will control the robot dog simply by typing everyday **English** (or Indonesian) commands such as `"Move forward"`, `"Turn left"`, or `"Sit down"`. An AI **Large Language Model (LLM)** translates your sentence into a friendly chat reply plus a hidden robot movement command, prints the exact JSON payload in the console for transparency, and politely refuses unsafe or impossible commands (like `"Fly to the ceiling"`) without moving the robot.

### Key Concepts for Beginners (No Prior Experience Needed)

- **What Is an LLM, System Prompt, Few-Shot Examples, and Temperature?**
  - **LLM (`qwen3.5:9b`)**: A text-understanding AI model that reads your typed command and turns it into structured robot numbers.
  - **System Prompt (`SYSTEM_PROMPT`)**: The AI's rulebook / instruction manual telling it what the robot can do, what number values to output, and how to reply.
  - **Few-Shot Examples (`FEW_SHOT_EXAMPLES`)**: Sample question-and-answer pairs shown to the AI before your command so it copies the exact output format reliably.
  - **Temperature (`0.0`)**: A "creativity" dial from `0.0` to `1.0+`. Setting `TEMPERATURE = 0.0` keeps answers consistent and predictable—important when controlling a physical robot!
- **What Are an API, API Key (`Bearer`), HTTP `POST`, and JSON Payload?**
  - **API** *(Application Programming Interface)*: Like a receptionist waiting for standardized order slips. This notebook talks to **two** APIs:
    1. **Cloud LLM API (`LLM_URL`)**: Hosted on the internet (`https://llm.fawzqi.my.id/v1/chat/completions`), protected by an **API Key** (`Authorization: Bearer b4workshop`—like a workshop passkey).
    2. **Local Robot API (`API_URL`)**: Running on the robot bridge (`/api/topics/publish`) to forward commands to the robot's motors.
  - **HTTP `POST` Request & JSON Payload**: Just like your browser loads websites over `http://`, sending an action command uses an **HTTP `POST`** request carrying a **JSON** order slip (curly braces `{...}` of key-value pairs):
    ```json
    {"topic": "/cmd_vel", "type": "geometry_msgs/Twist", "payload": {"linear": {"x": 1.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}}
    ```
- **What Are an IP Address, a Port, and `localhost`?**
  - **IP Address** (e.g., `192.168.1.100`): The robot's "street address" on the Wi-Fi network.
  - **Port** (e.g., `:3000`): The "room/door number" on that device where the robot API server listens.
  - **`localhost`**: A special nickname meaning **"this exact computer I am typing on right now"** (used when testing against a bridge or simulator on your own laptop).
- **How Can Google Colab Reach a Robot on Local Wi-Fi? (`BROWSER_LOCAL_FETCH = True`)**
  Google Colab's Python code runs on a Google Cloud computer far away—so Python can call the public **LLM API** (`LLM_URL`) directly, **but Python cannot reach `localhost` or `192.168.x.x` on your Wi-Fi**. With `BROWSER_LOCAL_FETCH = True` (already enabled), Colab passes the finished robot JSON back to **your web browser** on your laptop, and your browser delivers the HTTP `POST` straight over your local Wi-Fi to the robot!


## Block 1: Setup, Endpoint Config & Prompt Engineering

Run each grey code box in order by clicking its **Play (`▶`)** button (or pressing **`Shift + Enter`**) and waiting for the green checkmark (`✓`). This notebook uses only Python's built-in libraries, so no extra `pip install` is required!

### Beginner Step-by-Step Setup Checklist

1. **Join the Robot Wi-Fi & Check the Robot's Address**:
   - Connect your laptop to the workshop Wi-Fi and ask the instructor for the robot's **IP address** (e.g., `192.168.1.100`), or keep `localhost` if testing a local bridge/simulator on your own computer.
2. **Set `API_URL` in Step 1.1 Below**:
   - Change **only the address between `http://` and `:3000`**—keep `http://`, `:3000`, and `/api/topics/publish` exact:
   ```python
   BROWSER_LOCAL_FETCH = True
   API_URL = "http://192.168.1.100:3000/api/topics/publish"  # robot on local Wi-Fi (replace with your robot's IP)
   # or:
   API_URL = "http://localhost:3000/api/topics/publish"      # bridge running on your own laptop
   ```
3. **One-Time Browser Permission for a `http://192.168.x.x` Address** *(not needed for `localhost`)*:
   - Because Colab loads over `https://`, browsers block requests to plain `http://192.168.x.x` LAN addresses by default:
     1. Click the **tune / lock icon** at the left of the browser URL bar (next to `colab.research.google.com`).
     2. Click **Site settings**.
     3. Scroll to **Insecure content** and change it to **Allow**.
     4. Return to the Colab tab and click **Reload** when asked (if Chrome asks for **Local network access**, click **Allow**).
4. **Optional Advanced Mode (Public HTTPS Tunnel, `BROWSER_LOCAL_FETCH = False`)**:
   - Only needed if routing directly from Colab's cloud VM through an HTTPS tunnel (`ngrok http 3000` or `cloudflared tunnel --url http://localhost:3000`) to `https://<your-tunnel-domain>/api/topics/publish`.

### Robot Velocity Table (`geometry_msgs/Twist`)

| Field | Controls | Positive (`+`) | Negative (`-`) |
|---|---|---|---|
| `linear.x` | Forward / backward speed | Walk `forward` (`1.0`) | Walk `backward` (`-1.0`) |
| `linear.y` | Sideways step speed | Step `left` (`1.0`) | Step `right` (`-1.0`) |
| `angular.z` | Turning rotation speed | Turn `left` (`0.5`) | Turn `right` (`-0.5`) |
| `linear.z` | Posture height target | `stand` (`0.1`) | `sit` (`-0.1`) |
| *(all `0.0`)* | Safety stop | `stop` / halt / freeze | — |

### Step 1.1: Configure LLM Endpoint & Robot API Settings


In [ ]:
# =============================================================================
# 1. EASY CONFIGURATION (Edit these values)
# =============================================================================

LLM_URL = "https://llm.fawzqi.my.id/v1/chat/completions"  # Public OpenAI-compatible AI endpoint
LLM_API_KEY = "b4workshop"                                # Workshop passkey sent as Bearer token
MODEL_ID = "qwen3.5:9b"
REASONING_EFFORT = "none"                                 # Fast direct answers without hidden thinking tokens
TEMPERATURE = 0.0                                         # 0.0 = consistent, predictable robot commands

API_URL = "http://localhost:3000/api/topics/publish"      # <-- Change 'localhost' to the robot's Wi-Fi IP if needed (e.g. 192.168.1.100)
BROWSER_LOCAL_FETCH = True                                # True: your browser sends HTTP POST over local Wi-Fi; False: Colab Python sends it
CMD_VEL_TOPIC = "/cmd_vel"                                # Robot channel for velocity/motion commands
CMD_VEL_TYPE = "geometry_msgs/Twist"                      # ROS message structure expected by the robot bridge

print("Configuration ready. Robot API_URL:", API_URL)

### Step 1.2: Engineer Your System Prompt & Few-Shot Examples

Here you define **how the AI behaves**:
- **`SYSTEM_PROMPT`**: Sets the movement rules, exact `linear`/`angular` speeds, and refusal policy for impossible requests.
- **`FEW_SHOT_EXAMPLES`**: Shows the AI concrete sample conversations so it learns to reply cleanly with one friendly line followed by one valid JSON object—or a polite refusal with **no JSON**.

Try customizing speeds or adding your own sample commands below, then click **Play (`▶`)**.


In [ ]:
# =============================================================================
# 2. SYSTEM PROMPT ENGINEERING & FEW-SHOT EXAMPLES
# =============================================================================

# TODO: Engineer your system prompt here.
# Customize movement speeds, coordinate conventions, and response style.
SYSTEM_PROMPT = """
You are a friendly assistant that controls a robot dog (robodog).
Translate the user's command into a ROS geometry_msgs/Twist JSON message.

Response rules:
1. If the command contains a VALID action (forward, backward, move/strafe left, move/strafe right, turn/rotate left, turn/rotate right, stand/stand up, sit/sit down, stop/halt/freeze):
   - On the first line, reply with a short, friendly confirmation in English (for example: "Understood, moving forward."). If the user writes in Indonesian, you may reply in Indonesian.
   - On the next line, output EXACTLY ONE valid JSON object with NO markdown code fences and NO extra text after the JSON:
     {"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}
   - Velocity conventions:
     * forward / move forward / walk forward: linear.x = 1.0
     * backward / move backward / walk back: linear.x = -1.0
     * move left / strafe left / step left: linear.y = 1.0
     * move right / strafe right / step right: linear.y = -1.0
     * turn left / rotate left / spin left: angular.z = 0.5
     * turn right / rotate right / spin right: angular.z = -0.5
     * stand / stand up: linear.z = 0.1
     * sit / sit down / lie down: linear.z = -0.1
     * stop / halt / freeze / stay still: all values 0.0
2. If the command is INVALID, unsafe, impossible for a robot dog, or unclear:
   - Do NOT output any JSON at all.
   - Reply politely with: "I cannot do that." (or in the user's language).
""".strip()

# TODO: Add your few-shot examples here.
# Each entry pairs a sample "user" message with the expected "model" reply.
FEW_SHOT_EXAMPLES: list[dict[str, str]] = [
    {
        "user": "Move forward",
        "model": 'Understood, moving forward.\n{"linear": {"x": 1.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Walk backward",
        "model": 'Understood, walking backward.\n{"linear": {"x": -1.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Strafe left",
        "model": 'Understood, moving to the left.\n{"linear": {"x": 0.0, "y": 1.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Move to the right",
        "model": 'Understood, moving to the right.\n{"linear": {"x": 0.0, "y": -1.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Turn left",
        "model": 'Understood, turning left.\n{"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.5}}',
    },
    {
        "user": "Turn right",
        "model": 'Understood, turning right.\n{"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": -0.5}}',
    },
    {
        "user": "Stand up",
        "model": 'Understood, standing up.\n{"linear": {"x": 0.0, "y": 0.0, "z": 0.1}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Sit down",
        "model": 'Understood, sitting down.\n{"linear": {"x": 0.0, "y": 0.0, "z": -0.1}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Stop",
        "model": 'Understood, stopping now.\n{"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Fly up into the sky",
        "model": "I cannot do that.",
    },
    {
        "user": "Jump over the table",
        "model": "I cannot do that.",
    },
    {
        "user": "Chase that person",
        "model": "I cannot do that.",
    },
]

print(f"Prompt ready with {len(FEW_SHOT_EXAMPLES)} few-shot examples.")

### Step 1.3: Load LLM Client, JSON Extractor & Robot Publisher Helpers

Click **Play (`▶`)** to load the five core helper functions used behind the scenes:
- **`build_messages()`**: Packs `SYSTEM_PROMPT`, `FEW_SHOT_EXAMPLES`, and your command into an ordered chat list.
- **`ask_llm()`**: Sends that chat list to `LLM_URL` via an HTTP `POST` with your `Authorization: Bearer` key and returns the AI's raw text.
- **`split_reply_and_twist()`**: Separates the polite user-facing message from the JSON object (or returns `None` when the command was refused).
- **`build_robot_body()` / `publish_twist_payload()`**: Wraps the extracted Twist payload into `{"topic": "/cmd_vel", "type": "geometry_msgs/Twist", "payload": ...}` for your browser (or Python kernel) to send.


In [ ]:
import json
import re
import urllib.error
import urllib.request


def build_messages(
    user_text: str,
    system_prompt: str = SYSTEM_PROMPT,
    examples: list[dict[str, str]] = FEW_SHOT_EXAMPLES,
) -> list[dict[str, str]]:
    """Build chat messages with system prompt, few-shot examples, and user input."""
    messages = [{"role": "system", "content": system_prompt}]
    for example in examples:
        messages.append({"role": "user", "content": example["user"]})
        messages.append({"role": "assistant", "content": example["model"]})
    messages.append({"role": "user", "content": user_text})
    return messages


def ask_llm(
    user_text: str,
    model_id: str = MODEL_ID,
    llm_url: str = LLM_URL,
    api_key: str = LLM_API_KEY,
    reasoning_effort: str = REASONING_EFFORT,
    temperature: float = TEMPERATURE,
    timeout: float = 30.0,
) -> str:
    """Send a chat completion request to the LLM endpoint and return assistant text."""
    body = json.dumps({
        "model": model_id,
        "reasoning_effort": reasoning_effort,
        "temperature": temperature,
        "messages": build_messages(user_text),
    }).encode("utf-8")
    req = urllib.request.Request(
        llm_url,
        data=body,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "User-Agent": "robodog-workshop/1.0",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    return str(data["choices"][0]["message"]["content"]).strip()


def split_reply_and_twist(text: str) -> tuple[str, dict[str, dict[str, float]] | None]:
    """Separate user-facing reply text from an embedded Twist JSON payload."""
    decoder = json.JSONDecoder()
    idx = 0
    while (start := text.find("{", idx)) != -1:
        try:
            data, end_offset = decoder.raw_decode(text[start:])
        except json.JSONDecodeError:
            idx = start + 1
            continue

        if (
            isinstance(data, dict)
            and isinstance(data.get("linear"), dict)
            and isinstance(data.get("angular"), dict)
        ):
            twist = {
                section: {axis: float(data[section].get(axis, 0.0)) for axis in ("x", "y", "z")}
                for section in ("linear", "angular")
            }
            end = start + end_offset
            without_json = (text[:start] + text[end:]).strip()
            cleaned = re.sub(r"```(?:json)?\s*```", "", without_json).strip()
            return (cleaned or "Understood."), twist

        idx = start + 1

    return (text.strip() or "I cannot do that."), None


def build_robot_body(payload: dict[str, dict[str, float]]) -> dict:
    """Wrap a Twist payload into the REST bridge publish format."""
    return {
        "topic": CMD_VEL_TOPIC,
        "type": CMD_VEL_TYPE,
        "payload": payload,
    }


def publish_twist_payload(
    payload: dict[str, dict[str, float]],
    api_url: str = API_URL,
    timeout: float = 0.5,
) -> tuple[bool, str]:
    """POST a geometry_msgs/Twist JSON payload directly from the Python kernel."""
    body = json.dumps(build_robot_body(payload)).encode("utf-8")
    req = urllib.request.Request(
        api_url,
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return True, f"HTTP {resp.status}"
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        return False, f"HTTP error: {exc}"


print("Helpers loaded and ready.")

## Block 2: Running (Interactive Chat & Live Robot Control)

Click **Play (`▶`)** on the cell below to launch the interactive robot command console!

### How to Use the Console
1. Type a natural-language command into the box and press **Enter** (or click **Send**)—for example:
   - Valid tests: `Move forward`, `Turn left`, `Sit down`, `Stop`
   - Invalid test: `Fly to the ceiling` *(should politely refuse and send nothing to the robot)*
2. Check the console output lines after each command:
   - **`You: ...`**: What you typed.
   - **`Robot: ...`**: The friendly reply shown to the user (with raw JSON cleanly removed).
   - **`Payload: {"linear": ..., "angular": ...}`**: Exact Twist JSON payload sent to the robot (only printed when the command was valid).
   - **`Published to <API_URL> -> HTTP 200`** *(or **`Sent (no-cors)`**)*: Confirmation that your browser (or Python kernel) delivered the HTTP `POST` request.
3. Click **Clear** anytime to wipe the console log.

### Beginner Troubleshooting Guide
- **`LLM error: ...`**: Check your computer/Colab internet connection and make sure `LLM_URL` and `LLM_API_KEY` in Step 1.1 match the workshop values, then re-run Step 1.1–1.3.
- **`Published to ... -> HTTP err ... Failed to fetch`**: Your browser could not reach `API_URL`. Check that:
  1. The robot dog (and its REST API bridge on port `3000`) is powered on and your laptop is on the **same Wi-Fi**.
  2. `API_URL` in Step 1.1 has the exact right IP and port (e.g., `http://192.168.1.100:3000/api/topics/publish`), and you re-ran Step 1.1 + Block 2 after editing it.
  3. If `API_URL` uses `http://192.168.x.x`, you allowed **Insecure content** in Chrome **Site settings** for `colab.research.google.com` and reloaded the page.
- **AI misunderstands a command or picks the wrong speed**: Adjust the wording or add a matching example in `SYSTEM_PROMPT` / `FEW_SHOT_EXAMPLES` (Step 1.2), then re-run Step 1.2, Step 1.3, and Block 2.


In [ ]:
from google.colab import output
from IPython.display import HTML, JSON, display


def _llm_chat_cb(user_text: str):
    text = (user_text or "").strip()
    if not text:
        return JSON({"error": "Please enter a command."})
    try:
        reply = ask_llm(text)
    except (urllib.error.URLError, TimeoutError, OSError, KeyError, IndexError, json.JSONDecodeError) as exc:
        return JSON({"error": f"LLM error: {exc}"})

    user_message, payload = split_reply_and_twist(reply)
    publish_req = None
    api_status = None
    if payload is not None:
        if BROWSER_LOCAL_FETCH:
            publish_req = {"url": API_URL, "body": build_robot_body(payload)}
        else:
            _, api_status = publish_twist_payload(payload, api_url=API_URL)

    return JSON({
        "user_message": user_message,
        "payload": payload,
        "publish": publish_req,
        "api_url": API_URL,
        "api_status": api_status,
    })


output.register_callback("llm_chat_cb", _llm_chat_cb)

display(HTML("""
<div style="font-family:sans-serif; max-width:760px;">
  <div id="llm-status" style="margin-bottom:8px; font-weight:bold;">Ready! Type a robot command below.</div>
  <div id="llm-log" style="background:#181818; color:#eee; font-family:monospace; font-size:13px; padding:12px; height:280px; overflow-y:auto; border:1px solid #444; border-radius:4px; white-space:pre-wrap;"></div>
  <div style="display:flex; gap:8px; margin-top:8px;">
    <input id="llm-input" type="text" placeholder="Type a robot command (e.g. Move forward, Turn left, Sit down)..." style="flex:1; padding:8px; font-size:14px;" />
    <button id="llm-send" style="padding:8px 16px; background:#1976d2; color:#fff; border:none; cursor:pointer; font-weight:bold;">Send</button>
    <button id="llm-clear" style="padding:8px 12px; background:#555; color:#fff; border:none; cursor:pointer;">Clear</button>
  </div>
</div>
<script>
(() => {
  const statusEl = document.getElementById("llm-status");
  const logEl = document.getElementById("llm-log");
  const inputEl = document.getElementById("llm-input");
  const sendBtn = document.getElementById("llm-send");
  const clearBtn = document.getElementById("llm-clear");

  function appendLine(text, color = "#eee") {
    const line = document.createElement("div");
    line.style.color = color;
    line.textContent = text;
    logEl.appendChild(line);
    logEl.scrollTop = logEl.scrollHeight;
  }

  async function sendLocalApi(pub) {
    const payload = JSON.stringify(pub.body);
    try {
      const resp = await fetch(pub.url, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: payload,
      });
      return resp.ok ? "HTTP " + resp.status : "HTTP " + resp.status + " (error)";
    } catch (err) {
      try {
        await fetch(pub.url, {
          method: "POST",
          mode: "no-cors",
          headers: { "Content-Type": "text/plain" },
          body: payload,
        });
        return "Sent (no-cors)";
      } catch (err2) {
        return "HTTP err: " + err2;
      }
    }
  }

  async function submitCommand() {
    const cmd = inputEl.value.trim();
    if (!cmd) return;
    inputEl.value = "";
    sendBtn.disabled = true;
    statusEl.textContent = "Sending command to AI...";
    appendLine("\nYou: " + cmd, "#90caf9");
    try {
      const res = await google.colab.kernel.invokeFunction("llm_chat_cb", [cmd], {});
      let out = res?.data?.["application/json"];
      if (typeof out === "string") out = JSON.parse(out);
      if (!out || out.error) {
        appendLine(out?.error || "Unknown callback error", "#ef9a9a");
      } else {
        appendLine("Robot: " + out.user_message, "#a5d6a7");
        if (out.payload) {
          appendLine("Payload: " + JSON.stringify(out.payload), "#ffe082");
          if (out.publish) {
            const apiStatus = await sendLocalApi(out.publish);
            appendLine("Published to " + out.publish.url + " -> " + apiStatus, "#ce93d8");
          } else if (out.api_status) {
            appendLine("Published to " + out.api_url + " -> " + out.api_status, "#ce93d8");
          }
        }
      }
    } catch (err) {
      appendLine("Error: " + err, "#ef9a9a");
    } finally {
      sendBtn.disabled = false;
      statusEl.textContent = "Ready! Type a robot command below.";
      inputEl.focus();
    }
  }

  sendBtn.addEventListener("click", submitCommand);
  inputEl.addEventListener("keydown", (e) => {
    if (e.key === "Enter") submitCommand();
  });
  clearBtn.addEventListener("click", () => { logEl.textContent = ""; });
})();
</script>
"""))